# CastForge - ComfyUI on a rented GPU

One notebook, **Colab or Kaggle**, that stands ComfyUI up and hands your laptop a URL to talk to.

**Before you run anything, turn the GPU on.** Colab gives you a CPU-only machine by default, and
nothing here works on one.

- **Colab** - in the menu bar at the top: **Runtime > Change runtime type**. A box opens with
  "Hardware accelerator". Pick **T4 GPU**, then **Save**. (That is where "Tesla T4" comes from - it is
  the name of the card, and cell 1 prints it once the GPU is actually attached.)
- **Kaggle** - **Settings > Accelerator > GPU T4 x2** or P100, *and* **Internet > On**.

Then run the cells in order. The last one prints a `https://....trycloudflare.com` address. Give it to
the laptop:

```
python generate/remote_gpu.py --url https://xxxx.trycloudflare.com ping
python generate/remote_gpu.py --url https://xxxx.trycloudflare.com fabric "worn indigo denim, visible twill" --kind denim --name worn_denim
```

The laptop refuses a swatch with a seam in it, so you find that out here rather than on a character.

## 1 - Where are we, and is the GPU actually on?

In [ ]:
import os, pathlib, subprocess, sys

ON_KAGGLE = os.path.isdir("/kaggle")
ON_COLAB = os.path.isdir("/content") and not ON_KAGGLE
WORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
KEEP = None                      # somewhere a dropped session cannot take with it

print("running on:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "something else")
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    out = ""
if not out:
    raise SystemExit("No GPU. Turn the accelerator on first - see the note at the top.")
print(out)

## 2 - Somewhere the work survives a dropped session

Free sessions end: 90 minutes idle on Colab, twelve hours at the outside. Anything written only to the
instance dies with it. On Colab that means Drive; on Kaggle, `/kaggle/working`, which is kept when you
commit the notebook.

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    KEEP = pathlib.Path("/content/drive/MyDrive/castforge")
else:
    KEEP = WORK / "castforge"
(KEEP / "swatches").mkdir(parents=True, exist_ok=True)
print("keeping finished work in", KEEP)

## 3 - ComfyUI, the Manager, and the one node that makes a texture tile

In [ ]:
COMFY = WORK / "ComfyUI"
if not COMFY.exists():
    !git clone -q https://github.com/comfyanonymous/ComfyUI {COMFY}
%cd {COMFY}
!pip install -q -r requirements.txt

NODES = COMFY / "custom_nodes"

def grab(url):
    name = url.rstrip("/").split("/")[-1]
    if not (NODES / name).exists():
        !git clone -q {url} {NODES / name}
    print("node pack:", name)

grab("https://github.com/ltdrdata/ComfyUI-Manager")
# Without this one, a generated square has a hard join and every repeat of it shows a seam.
# remote_gpu.py refuses such a swatch, so do not skip it.
grab("https://github.com/spinagon/ComfyUI-seamless-tiling")

## 4 - A checkpoint, downloaded ONCE and kept on Drive

Nothing is pre-installed on a Colab machine: every session starts empty. So the checkpoint is
downloaded into **Drive**, not into the session, and every run after the first one just links to it.
One 6.5 GB download, ever.

The default is **RealVisXL V5.0 Lightning** - the same checkpoint `comfyui-graphic-novel` already uses,
which is the point of picking it. It is photoreal, which is what cloth needs, and it is a Lightning
model: **6 steps instead of 28**, so a swatch takes about fifteen seconds on a free T4 rather than a
minute and a half. It is not gated, so no token is needed for it.

A gated model (FLUX.1-dev) needs a Hugging Face token. Put it in Colab's **Secrets** panel - the key
icon in the left margin - named `HF_TOKEN`, with "Notebook access" on. Never paste a token into a cell:
anything typed into a cell is saved in the notebook and goes wherever the notebook goes.

In [ ]:
CHECKPOINTS = {
    # name on disk: (repo, file inside it, needs a token?)
    "realvis_lightning": ("SG161222/RealVisXL_V5.0_Lightning",
                          "RealVisXL_V5.0_Lightning_fp16.safetensors", False),
    "sdxl_base": ("stabilityai/stable-diffusion-xl-base-1.0",
                  "sd_xl_base_1.0.safetensors", False),
    "flux_dev": ("black-forest-labs/FLUX.1-dev", "flux1-dev.safetensors", True),
}
WANT = "realvis_lightning"        # change this if you want a different one

repo, filename, gated = CHECKPOINTS[WANT]
STORE = KEEP / "models" / "checkpoints"        # on Drive: survives the session
STORE.mkdir(parents=True, exist_ok=True)
kept = STORE / filename

token = None
if gated:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = os.environ.get("HF_TOKEN")
    if not token:
        raise SystemExit("%s is gated. Put a token in the Secrets panel (key icon) as HF_TOKEN, "
                         "with Notebook access on." % repo)

if kept.exists():
    print("already on Drive: %s (%.1f GB) - nothing to download" % (kept.name,
                                                                    kept.stat().st_size / 2 ** 30))
else:
    print("first time: downloading %s into Drive. This happens once." % filename)
    !pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download
    got = hf_hub_download(repo_id=repo, filename=filename, token=token,
                          local_dir=str(STORE), local_dir_use_symlinks=False)
    print("downloaded to", got)

MODELS = COMFY / "models" / "checkpoints"
MODELS.mkdir(parents=True, exist_ok=True)
link = MODELS / filename
if not link.exists():
    link.symlink_to(kept)          # ComfyUI reads it straight off Drive
print("ComfyUI will load:", filename)
print("tell the laptop:  --checkpoint", filename)

## 5 - Start it, and open a door to it

`cloudflared` needs no account and no signup, and the address it prints works from your laptop straight
away. It is a NEW address every time you run this cell - paste the new one into `--url`.

In [ ]:
import re, threading, time, urllib.request

if not pathlib.Path("/usr/local/bin/cloudflared").exists():
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

comfy = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
                          "--output-directory", str(KEEP / "swatches")],
                         cwd=str(COMFY), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

up = False
for _ in range(90):                        # answer first, then open the door
    time.sleep(2)
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3)
        up = True
        break
    except Exception:
        pass
if not up:
    raise SystemExit("ComfyUI did not start - run cell 3 again and look for install errors")
print("ComfyUI is up")

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188",
                           "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in tunnel.stdout:
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        print()
        print("=" * 70)
        print("  Point the laptop at this:")
        print()
        print("  python generate/remote_gpu.py --url %s ping" % found.group(0))
        print("=" * 70)
        print()
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

## 6 - Keep the session alive while you work

A free session with nothing happening in the browser is a session about to end. Run this and leave the
tab open; stop it when you are finished.

In [ ]:
import datetime, time

# No carriage return anywhere in here. A notebook keeps its source as JSON, and a real CR inside a
# line splits that line when the file is read back, which breaks the whole cell.
try:
    while True:
        print("alive", datetime.datetime.now().strftime("%H:%M:%S"),
              "- stop this cell when you have finished", flush=True)
        time.sleep(300)
except KeyboardInterrupt:
    print("stopped")

---
### When it goes wrong

| what you see | what it means |
|---|---|
| `seamless tiling NOT installed` from `ping` | cell 3 did not finish - run it again |
| `the best of 3 still has a seam` | the node is installed but not being used: check `CircularVAEDecode` is in what `ping` lists |
| the address stops answering | the session dropped. Re-run cell 5 and use the NEW address |
| `cannot reach a ComfyUI` | cell 5 is not running, or you pasted the old address |
| `is gated` in cell 4 | put a Hugging Face token in the Secrets panel (key icon) as `HF_TOKEN`, Notebook access on - or leave `WANT` on `realvis_lightning`, which needs none |
| cell 4 downloads again every session | Drive did not mount. Run cell 2 and click through the popup |

Finished swatches are kept in `castforge/swatches` on Drive (Colab) or `/kaggle/working/castforge/swatches`
(Kaggle), so a dropped session never loses one that was already made.